
# 1. Business Understanding

## 1.1 Project Background

The project focuses on the Brazilian Olist e-commerce marketplace.

The objective is to understand the relationship between delivery
performance, customer satisfaction and sales.

The analysis uses historical Olist e-commerce data from
January 2017 to August 2018.

## 1.2 Business Problem

Olist has data distributed across multiple source files.
There is no single trusted view of delivery performance,
customer satisfaction and sales.

The relationship between late deliveries and customer
review scores needs to be investigated.

## 1.3 Main Research Question

How is delivery performance associated with customer satisfaction and sales outcomes in the Olist e-commerce marketplace?

        - The analysis describes relationships in historical observational data. 
        - It does not by itself prove that delivery delays cause changes in reviews or sales.

## 1.4 Supporting Business Questions

1. How many orders fall within the agreed analysis period, and how does order volume vary by status and month?
2. What is the GMV (item price excluding freight), and what is the average order value (payment value divided by eligible order count)?
3. What proportion of delivered orders arrive on or before the promised date?
4. How long does delivery take from purchase to customer receipt?
5. How many days late are orders that arrive after the promised date?
6. What proportion of eligible orders meet the seller dispatch deadline?
7. What is the average difference between the promised and actual delivery dates?
8. What is the average customer review score?
9. What proportion of reviews are poor (1–2 stars)?
10. How different are average review scores for on-time and late deliveries?
11. What proportion of unique customers place two or more orders in the analysis period?

    Monthly, state and product-category breakdowns can be used as analysis dimensions where the data and sample sizes support them.

## 1.5 Analysis Objectives

- Measure delivery performance.
- Identify delivery delays and patterns.
- Analyse customer review scores.
- Examine the relationship between delivery performance
  and customer satisfaction.
- Compare sales indicators across delivery groups.
- Develop useful business insights.

## 1.6 Project Scope

Dataset: Brazilian Olist e-commerce dataset.

Analysis period: January 2017 to August 2018.

Tools: Python, MySQL and Power BI.

Source data: Nine original CSV files.



## 1.7 Initial KPI Framework

| ID | KPI | Formula supplied by Aimen | Reported baseline | Proposed target |
|---|---|---|---:|---|
| K01 | GMV | SUM(item price), excluding freight | 13,449,529.68 BRL | +10% YoY |
| K02 | Number of orders | Distinct order count, excluding cancelled/unavailable | 97,905 | +10% YoY |
| K03 | Average order value | SUM(payment value) / eligible order count | 160.22 BRL | >= 165 BRL |
| K04 | On-time delivery rate | Delivered on or before estimate / all delivered orders | 93.21% | >= 95% |
| K05 | Average delivery time | Average actual delivery date minus purchase timestamp | 12.54 days | <= 10 days |
| K06 | Average delay of late orders | Average actual delivery date minus estimated date, for late orders | 10.62 days | <= 7 days |
| K07 | Seller dispatch compliance | Orders handed to carrier by shipping deadline / delivered orders | 91.08% | >= 95% |
| K08 | Delivery estimate buffer | Average estimated date minus actual delivery date | 11.81 days | 5–8 days |
| K09 | Average review score | Average review score using latest review per order | 4.09 | >= 4.2 |
| K10 | Poor review rate | Reviews with score <= 2 / all reviews | 14.63% | <= 12% |
| K11 | Review gap: on-time vs late | Average on-time score minus average late score | 2.02 stars | Monitor |
| K12 | Repeat customer rate | Unique customers with 2+ orders / all unique customers | 3.11% | >= 5% |

**Validation note:** Reported KPI baselines are provisional
until independently recalculated. The calculation rules for
K07 and the review population for K09–K11 will be confirmed
before final reporting.

# 2. Data Loading

## 2.1 Objective

The objective of this stage is to load the nine original Olist
CSV files into Python using Pandas.

The datasets will be checked for file availability, successful
loading, row counts, column counts and basic structure.

The original source files will remain unchanged.

In [140]:
from pathlib import Path

# Find the Olist project folder
current_folder = Path.cwd().resolve()

PROJECT_ROOT = None

for folder in [current_folder, *current_folder.parents]:
    if (folder / "02_data_acquisition" / "raw_data").is_dir():
        PROJECT_ROOT = folder
        break

# Fallback to the known project location
if PROJECT_ROOT is None:
    possible_path = (
        Path.home()
        / "Documents/University/BI_BigData/Olist_Project"
    )

    if (possible_path / "02_data_acquisition" / "raw_data").is_dir():
        PROJECT_ROOT = possible_path

if PROJECT_ROOT is None:
    raise FileNotFoundError("Olist project folder not found.")

# Define the data location
DATA_DIR = PROJECT_ROOT / "02_data_acquisition" / "raw_data"

print("Project location:", PROJECT_ROOT)
print("Data location:", DATA_DIR)
print("Data folder exists:", DATA_DIR.exists())

Project location: /Users/tarun/Documents/University/BI_BigData/Olist_Project
Data location: /Users/tarun/Documents/University/BI_BigData/Olist_Project/02_data_acquisition/raw_data
Data folder exists: True


## 2.3 Check the available CSV files

In [141]:
csv_files = sorted(DATA_DIR.glob("*.csv"))

print("Number of CSV files:", len(csv_files))

for file in csv_files:
    print(file.name)

Number of CSV files: 9
olist_customers_dataset.csv
olist_geolocation_dataset.csv
olist_order_items_dataset.csv
olist_order_payments_dataset.csv
olist_order_reviews_dataset.csv
olist_orders_dataset.csv
olist_products_dataset.csv
olist_sellers_dataset.csv
product_category_name_translation.csv


## 2.4 Load the nine datasets into Python

In this step, we use Pandas to load all nine CSV files
into memory as separate DataFrames.

Each DataFrame represents one source table.
The original CSV files will remain unchanged.

In [142]:

import pandas as pd
import numpy as np

# Define the nine source files
DATA_FILES = {
    "customers": "olist_customers_dataset.csv",
    "geolocation": "olist_geolocation_dataset.csv",
    "order_items": "olist_order_items_dataset.csv",
    "order_payments": "olist_order_payments_dataset.csv",
    "order_reviews": "olist_order_reviews_dataset.csv",
    "orders": "olist_orders_dataset.csv",
    "products": "olist_products_dataset.csv",
    "sellers": "olist_sellers_dataset.csv",
    "category_translation": "product_category_name_translation.csv"
}

# Create an empty dictionary
data = {}

# Load each CSV file
for dataset_name, filename in DATA_FILES.items():

    file_path = DATA_DIR / filename

    data[dataset_name] = pd.read_csv(
        file_path,
        low_memory=False
    )

    print(f"Successfully loaded: {dataset_name}")

print("\nTotal datasets loaded:", len(data))


Successfully loaded: customers
Successfully loaded: geolocation
Successfully loaded: order_items
Successfully loaded: order_payments
Successfully loaded: order_reviews
Successfully loaded: orders
Successfully loaded: products
Successfully loaded: sellers
Successfully loaded: category_translation

Total datasets loaded: 9


## 2.5 Dataset Overview

This section summarises the number of records and
columns in each loaded dataset.

In [143]:
dataset_summary = []

for name, df in data.items():

    dataset_summary.append({
        "Dataset": name,
        "Rows": df.shape[0],
        "Columns": df.shape[1],
        "Memory (MB)": round(
            df.memory_usage(deep=True).sum() / (1024 ** 2), 2
        )
    })

dataset_summary_df = pd.DataFrame(dataset_summary)

display(dataset_summary_df)

,Dataset,Rows,Columns,Memory (MB)
0,customers,99441,5,26.59
1,geolocation,1000163,5,129.38
2,order_items,112650,7,35.99
3,order_payments,103886,5,16.23
4,order_reviews,99224,7,39.12
5,orders,99441,8,52.94
6,products,32951,9,6.30
7,sellers,3095,4,0.59
8,category_translation,71,2,0.01


## 2.6 Initial Inspection of the Orders Dataset

The orders dataset contains order information, order status,
purchase timestamps and delivery timestamps.

It will be a key source for analysing delivery performance.

In [144]:
# Display the first five records
display(data["orders"].head())

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00


In [145]:
# Display the column names
print("Columns in the orders dataset:")

for column in data["orders"].columns:
    print(column)

Columns in the orders dataset:
order_id
customer_id
order_status
order_purchase_timestamp
order_approved_at
order_delivered_carrier_date
order_delivered_customer_date
order_estimated_delivery_date


In [146]:
# Check the data types
data["orders"].dtypes

order_id                         str
customer_id                      str
order_status                     str
order_purchase_timestamp         str
order_approved_at                str
order_delivered_carrier_date     str
order_delivered_customer_date    str
order_estimated_delivery_date    str
dtype: object

## 2.7 Initial Order Status Overview

This section examines the different order statuses
recorded in the Olist orders dataset.

In [147]:
# Count orders by status
order_status_summary = (
    data["orders"]["order_status"]
    .value_counts(dropna=False)
)

display(order_status_summary)

order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64

In [148]:
# Create reports folder
REPORT_DIR = PROJECT_ROOT / "04_eda" / "reports"

REPORT_DIR.mkdir(parents=True, exist_ok=True)

# Save dataset overview
summary_file = REPORT_DIR / "02_data_loading_summary.csv"

dataset_summary_df.to_csv(summary_file, index=False)

print("Dataset summary successfully saved!")
print("File location:", summary_file)

Dataset summary successfully saved!
File location: /Users/tarun/Documents/University/BI_BigData/Olist_Project/04_eda/reports/02_data_loading_summary.csv


# 3. Initial Data Exploration

## 3.1 Objective

The objective of this stage is to understand the structure
and business relevance of the nine Olist datasets.

We will identify important tables, columns and variables
required for analysing delivery performance, customer
satisfaction and sales.

Detailed data dictionary and data quality profiling
will be coordinated with Aimen's work.

In [149]:
# List the columns available in each dataset

for name, df in data.items():

    print("\n" + "=" * 60)
    print("DATASET:", name.upper())
    print("=" * 60)

    print("Number of rows:", df.shape[0])
    print("Number of columns:", df.shape[1])

    print("\nAvailable columns:")

    for column in df.columns:
        print("-", column)


DATASET: CUSTOMERS
Number of rows: 99441
Number of columns: 5

Available columns:
- customer_id
- customer_unique_id
- customer_zip_code_prefix
- customer_city
- customer_state

DATASET: GEOLOCATION
Number of rows: 1000163
Number of columns: 5

Available columns:
- geolocation_zip_code_prefix
- geolocation_lat
- geolocation_lng
- geolocation_city
- geolocation_state

DATASET: ORDER_ITEMS
Number of rows: 112650
Number of columns: 7

Available columns:
- order_id
- order_item_id
- product_id
- seller_id
- shipping_limit_date
- price
- freight_value

DATASET: ORDER_PAYMENTS
Number of rows: 103886
Number of columns: 5

Available columns:
- order_id
- payment_sequential
- payment_type
- payment_installments
- payment_value

DATASET: ORDER_REVIEWS
Number of rows: 99224
Number of columns: 7

Available columns:
- review_id
- order_id
- review_score
- review_comment_title
- review_comment_message
- review_creation_date
- review_answer_timestamp

DATASET: ORDERS
Number of rows: 99441
Number of 

## 3.2 Identify Key Variables for Business Analysis

This section identifies the important variables needed
to investigate delivery performance, customer satisfaction
and sales.

The selection is based on the data dictionary prepared
by Aimen.

In [150]:

# Key variables required for our business analysis

key_variables = {
    "orders": [
        "order_id",
        "customer_id",
        "order_status",
        "order_purchase_timestamp",
        "order_delivered_carrier_date",
        "order_delivered_customer_date",
        "order_estimated_delivery_date"
    ],

    "order_reviews": [
        "order_id",
        "review_score",
        "review_creation_date",
        "review_answer_timestamp"
    ],

    "order_items": [
        "order_id",
        "order_item_id",
        "product_id",
        "seller_id",
        "shipping_limit_date",
        "price",
        "freight_value"
    ],

    "order_payments": [
        "order_id",
        "payment_sequential",
        "payment_type",
        "payment_installments",
        "payment_value"
    ],

    "customers": [
        "customer_id",
        "customer_unique_id",
        "customer_zip_code_prefix",
        "customer_city",
        "customer_state"
    ],

    "sellers": [
        "seller_id",
        "seller_zip_code_prefix",
        "seller_city",
        "seller_state"
    ],

    "products": [
        "product_id",
        "product_category_name"
    ],

    "category_translation": [
        "product_category_name",
        "product_category_name_english"
    ],

    "geolocation": [
        "geolocation_zip_code_prefix",
        "geolocation_lat",
        "geolocation_lng",
        "geolocation_city",
        "geolocation_state"
    ]
}

# Check whether the selected columns exist
variable_check = []

for dataset, columns in key_variables.items():

    for column in columns:

        variable_check.append({
            "Dataset": dataset,
            "Variable": column,
            "Available": column in data[dataset].columns
        })

variable_check_df = pd.DataFrame(variable_check)

display(variable_check_df)

print(
    "Total variables checked:",
    len(variable_check_df)
)

print(
    "Missing variable names:",
    (~variable_check_df["Available"]).sum()
)


,Dataset,Variable,Available
0,orders,order_id,True
1,orders,customer_id,True
2,orders,order_status,True
3,orders,order_purchase_timestamp,True
4,orders,order_delivered_carrier_date,True
5,orders,order_delivered_customer_date,True
6,orders,order_estimated_delivery_date,True
7,order_reviews,order_id,True
8,order_reviews,review_score,True
9,order_reviews,review_creation_date,True


Total variables checked: 41
Missing variable names: 0
